In [1]:
from pathlib import Path
import urllib.request

def download_shakespeare_text():
  path = Path("datasets/shakespeare/shakespeare.txt")
  if not path.is_file():
    path.parent.mkdir(parents = True, exist_ok = True)
    url = "https://homl.info/shakespeare"
    urllib.request.urlretrieve(url, path)
  return path.read_text()

In [2]:
shakespeare_text = download_shakespeare_text()

In [3]:
print(shakespeare_text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [4]:
vocab = sorted(set(shakespeare_text.lower()))
"".join(vocab)

"\n !$&',-.3:;?abcdefghijklmnopqrstuvwxyz"

In [5]:
char_to_id = {char: index for index, char in enumerate(vocab)}
id_to_char = {index: char for index, char in enumerate(vocab)}
char_to_id["a"]

13

In [6]:
id_to_char[15]
# we converted the characters in to char to id and id to char with index and values switched

'c'

In [7]:
import torch
def encode_text(text):
  return torch.tensor([char_to_id[char] for char in text.lower()])

def decode_text(char_ids):
  return "".join([id_to_char[char_id.item()] for char_id in char_ids])

In [8]:
encoded = encode_text("Hello, World!")
encoded

tensor([20, 17, 24, 24, 27,  6,  1, 35, 27, 30, 24, 16,  2])

In [9]:
decode_text(encoded)

# We used the shakespeare text as the corpus to encode and decode the text by using its char id index

'hello, world!'

In [10]:
from torch.utils.data import Dataset, DataLoader

class CharDataset(Dataset):

  def __init__(self, text, window_length):
    self.encoded_text = encode_text(text)
    self.window_length = window_length

  def __len__(self):
    return len(self.encoded_text) - self.window_length

  def __getitem__(self, idx):
    if idx >= len(self):
      raise IndexError("dataset Index out of Range")
    end = idx + self.window_length
    window = self.encoded_text[idx : end]
    target = self.encoded_text[idx + 1 : end + 1]
    return window, target

# We created custom dataset with window length which is used ot predict the end index next value by seperating window data and target data for prediction in sequence

In [11]:
window_length = 50
batch_size = 512
train_set = CharDataset(shakespeare_text[:1_000_000], window_length)
valid_set = CharDataset(shakespeare_text[1_000_000:1_060_000], window_length)
test_set = CharDataset(shakespeare_text[1_060_000:], window_length)
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_set, batch_size=batch_size)
test_loader = DataLoader(test_set, batch_size=batch_size)

In [12]:
import torch.nn as nn
torch.manual_seed(42)
embed = nn.Embedding(5, 3)
# We used nn.EMbedding to create a embedding dimension with (num_embeddings, embedding dimension)
embed(torch.tensor([[3, 2], [0, 2]]))

# here we can see that the second vector in both matrix is same as we given it as 2 (same meaning)

tensor([[[ 0.2674,  0.5349,  0.8094],
         [ 2.2082, -0.6380,  0.4617]],

        [[ 0.3367,  0.1288,  0.2345],
         [ 2.2082, -0.6380,  0.4617]]], grad_fn=<EmbeddingBackward0>)

In [34]:
import torchmetrics

def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()

def train(model, optimizer, loss_fn, metric, train_loader, valid_loader,
          n_epochs, patience=2, factor=0.5, epoch_callback=None):
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", patience=patience, factor=factor)
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()
        if epoch_callback is not None:
            epoch_callback(model, epoch)
        for index, (X_batch, y_batch) in enumerate(train_loader):
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
            train_metric = metric.compute().item()
            print(f"\rBatch {index + 1}/{len(train_loader)}", end="")
            print(f", loss={total_loss/(index+1):.4f}", end="")
            print(f", {train_metric=:.2%}", end="")
        history["train_losses"].append(total_loss / len(train_loader))
        history["train_metrics"].append(train_metric)
        val_metric = evaluate_tm(model, valid_loader, metric).item()
        history["valid_metrics"].append(val_metric)
        scheduler.step(val_metric)
        print(f"\rEpoch {epoch + 1}/{n_epochs},                      "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.2%}, "
              f"valid metric: {history['valid_metrics'][-1]:.2%}")
    return history

In [35]:
device = "cuda" if torch.cuda.is_available() else "cpu"
class ShakespeareModel(nn.Module):
    def __init__(self, vocab_size, n_layers=2, embed_dim=10, hidden_dim=128,
                 dropout=0.1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers,
                          batch_first=True, dropout=dropout)
        self.output = nn.Linear(hidden_dim, vocab_size)

    def forward(self, X):
        embeddings = self.embed(X)
        outputs, _states = self.gru(embeddings)
        return self.output(outputs).permute(0, 2, 1)

torch.manual_seed(42)
model = ShakespeareModel(len(vocab)).to(device)

In [36]:
import torchmetrics
device = "cuda" if torch.cuda.is_available() else "cpu"

n_epochs = 20
xentropy = nn.CrossEntropyLoss()
optimizer = torch.optim.NAdam(model.parameters())
accuracy = torchmetrics.Accuracy(task="multiclass",
                                 num_classes=len(vocab)).to(device)

history = train(model, optimizer, xentropy, accuracy, train_loader, valid_loader,
                n_epochs)

Epoch 1/20,                      train loss: 1.6040, train metric: 51.28%, valid metric: 51.98%
Epoch 2/20,                      train loss: 1.3843, train metric: 56.72%, valid metric: 52.83%
Epoch 3/20,                      train loss: 1.3547, train metric: 57.46%, valid metric: 53.64%
Epoch 4/20,                      train loss: 1.3403, train metric: 57.81%, valid metric: 53.45%
Epoch 5/20,                      train loss: 1.3320, train metric: 58.02%, valid metric: 53.32%
Epoch 6/20,                      train loss: 1.3264, train metric: 58.16%, valid metric: 53.79%
Epoch 7/20,                      train loss: 1.3225, train metric: 58.26%, valid metric: 53.71%
Epoch 8/20,                      train loss: 1.3193, train metric: 58.34%, valid metric: 54.09%
Epoch 9/20,                      train loss: 1.3167, train metric: 58.40%, valid metric: 54.33%
Epoch 10/20,                      train loss: 1.3148, train metric: 58.45%, valid metric: 54.08%
Epoch 11/20,                      train

In [37]:
torch.save(model.state_dict(), "my_shakespeare_model.pt")

In [38]:
model.eval()
text = "To be or not to b"
encoded_text = encode_text(text).unsqueeze(dim=0).to(device)
with torch.no_grad():
 Y_logits = model(encoded_text)
 predicted_char_id = Y_logits[0, :, -1].argmax().item()
 predicted_char = id_to_char[predicted_char_id]

In [39]:
predicted_char

'e'

In [40]:
torch.manual_seed(42)
probs = torch.tensor([[0.5, 0.4, 0.1]])
samples = torch.multinomial(probs, replacement=True, num_samples=8)
samples


tensor([[0, 0, 0, 0, 1, 0, 2, 2]])

In [41]:
import torch.nn.functional as F

def next_char(model, text, temperature=1):
    encoded_text = encode_text(text).unsqueeze(dim=0).to(device)
    with torch.no_grad():
        Y_logits = model(encoded_text)
        Y_probas = F.softmax(Y_logits[0, :, -1] / temperature, dim=-1)
        predicted_char_id = torch.multinomial(Y_probas, num_samples=1).item()
    return id_to_char[predicted_char_id]

In [42]:
def extend_text(model, text, n_chars=80, temperature=1):
  for _ in range(n_chars):
    text += next_char(model, text, temperature)
  return text

# To generate text each word after using the next_char function by changing temperature to get different kind of sequences

In [43]:
print(extend_text(model, "To be or not to b", temperature = 0.01))
# Repitive when the temperature is low

To be or not to be the county and the state
that thou hast should not stand the county and the co


In [44]:
print(extend_text(model, "To be or not to b", temperature = 0.4))

To be or not to be so fare
the people that i have respected thee them first,
that he had the matt


In [45]:
print(extend_text(model, "To be or not to b", temperature=100))
# Very random when the temperature is high

To be or not to bmhf:my:r,k;s-h cqvvnfnfsut&-oq'ryoeen?x-hp:d,y&wv f3,dzrdzj-pilv?xpzc,fborp;'?$u
